# Computational Graphs: Static vs Dynamic

**Author:** Phabel Antonio López Delgado, BSc.

This notebook explores the concept of **computational graphs**, which model mathematical functions as directed graphs of operations on tensors. Computational graphs are fundamental to deep learning frameworks, as they enable automatic differentiation (backpropagation).

We compare two approaches:

- **Static Graphs** (TensorFlow 1.x) – the graph is defined once and executed later.
- **Dynamic Graphs** (PyTorch) – the graph is built and executed on the fly.

**Key Concepts Covered:**
- Computational graph structure (nodes: operations, edges: data flow).
- Forward pass: computing the output.
- Backward pass: computing gradients via backpropagation.
- Static vs dynamic graph trade‑offs.

**Key Techniques & Libraries:**
- `tensorflow.compat.v1` – static graph definition and execution.
- `torch` – dynamic graph and automatic differentiation.

**Objective:**
To understand how computational graphs work and the differences between static and dynamic execution, which underpins the design of modern deep learning frameworks.

In [1]:
import tensorflow.compat.v1 as tf
import torch

# Disable TensorFlow v2 behaviour to use v1 syntax
tf.disable_v2_behavior()

print("All imports successful.")


Instructions for updating:
non-resource variables are not supported in the long term
All imports successful.


## 1. Static Computational Graphs (TensorFlow 1.x)

In a static graph, we first define the entire graph structure (placeholders, operations) and then execute it within a session. The graph is fixed and cannot be changed during execution.

### Example Function

We compute:

$a = y - z$

$b = x * a^2$


We want the output `b` and its gradients with respect to `x`, `y`, and `z`.

### Steps:

1. Define placeholders for inputs (`x`, `y`, `z`).
2. Define operations (`a`, `b`).
3. Create a session and run the graph with a `feed_dict`.
4. Compute gradients using `tf.gradients()`.

In [6]:
def static_graph(x1, x2, x3):
    # Placeholders: input nodes
    x = tf.placeholder(tf.float32)
    y = tf.placeholder(tf.float32)
    z = tf.placeholder(tf.float32)

    # Operations
    a = y - z
    b = x * a ** 2

    # Dictionary mapping placeholders to actual values
    feed_dict = {x: [x1], y: [x2], z: [x3]}

    # Run the graph in a session
    with tf.Session() as sess:
        # Forward pass: compute b (extract scalar with [0])
        forward_out = sess.run(b, feed_dict=feed_dict)[0]

        # Backward pass: compute gradients (also extract scalars)
        dx = sess.run(tf.gradients(b, x), feed_dict=feed_dict)[0][0]
        dy = sess.run(tf.gradients(b, y), feed_dict=feed_dict)[0][0]
        dz = sess.run(tf.gradients(b, z), feed_dict=feed_dict)[0][0]

    print(f"Forward output b = {forward_out:.4f}")
    print(f"Gradients: db/dx = {dx:.4f}, db/dy = {dy:.4f}, db/dz = {dz:.4f}")
    return forward_out, (dx, dy, dz)

In [7]:
print("Static Graph Example:")
static_graph(1.0, 4.4, 3.0)

Static Graph Example:
Forward output b = 1.9600
Gradients: db/dx = 1.9600, db/dy = 2.8000, db/dz = -2.8000


(1.9600003, (1.9600003, 2.8000002, -2.8000002))

## 2. Dynamic Computational Graphs (PyTorch)

In a dynamic graph, operations are executed immediately as they are defined. The graph is built on the fly and can change dynamically. This is known as **define‑by‑run**.

### Same Example

We compute the same function `b = x * (y - z)^2` using PyTorch. Steps:

1. Create tensors with `requires_grad=True` to track gradients.
2. Perform operations (forward pass).
3. Call `.backward()` to compute gradients.
4. Access gradients via `.grad` attribute.

No session or feed_dict is required – the computation happens immediately.

In [8]:
def dynamic_graph(x1, x2, x3):
    # Tensors with gradient tracking
    x = torch.tensor(x1, dtype=torch.float32, requires_grad=True)
    y = torch.tensor(x2, dtype=torch.float32, requires_grad=True)
    z = torch.tensor(x3, dtype=torch.float32, requires_grad=True)

    # Forward pass
    a = y - z
    b = x * a ** 2

    # Backward pass (compute gradients)
    b.backward()

    # Extract gradients
    dx = x.grad.item()
    dy = y.grad.item()
    dz = z.grad.item()

    print(f"Forward output b = {b.item():.4f}")
    print(f"Gradients: db/dx = {dx:.4f}, db/dy = {dy:.4f}, db/dz = {dz:.4f}")
    return b.item(), (dx, dy, dz)

In [9]:
print("Dynamic Graph Example:")
dynamic_graph(2.0, 5.0, 3.0)

Dynamic Graph Example:
Forward output b = 8.0000
Gradients: db/dx = 4.0000, db/dy = 8.0000, db/dz = -8.0000


(8.0, (4.0, 8.0, -8.0))

## 3. Comparison of Static and Dynamic Graphs

| Feature                | Static Graph (TF1)            | Dynamic Graph (PyTorch)        |
|------------------------|-------------------------------|--------------------------------|
| Definition             | Define once, execute later    | Define and execute on the fly  |
| Debugging              | Harder (session required)     | Easier (native Python debugger)|
| Flexibility            | Fixed structure               | Can change dynamically         |
| Performance            | Optimised for inference       | Slightly slower but improving  |
| Use Cases              | Production deployment         | Research and prototyping       |
| Gradient Computation   | Manual with `tf.gradients()`  | Automatic via `.backward()`    |

**Key Insight:** Most modern frameworks (PyTorch, TensorFlow 2.x) use dynamic graphs by default because they offer greater flexibility and ease of debugging, while static graphs are still used in production for performance optimisation.

## Summary and Next Steps

**Accomplished:**
- Implemented a computational graph using TensorFlow 1.x (static graph).
- Implemented the same computation using PyTorch (dynamic graph).
- Compared forward and backward passes in both frameworks.

**Key Results:**
- Both frameworks compute the same forward output and gradients.
- Static graphs require a session and feed_dict; dynamic graphs execute immediately.
- Gradients are accessible via `.grad` in PyTorch and via `tf.gradients()` in TensorFlow.

**Key Insights:**
- Computational graphs are the foundation of automatic differentiation.
- Dynamic graphs are more intuitive and easier to debug, making them preferred for research.
- Static graphs offer performance benefits for large‑scale deployment.

**Suggested Next Steps:**
1. **Explore TensorFlow 2.x** – which uses eager execution (dynamic) by default.
2. **Build a multi‑layer neural network** – using computational graphs and backpropagation.
3. **Visualise computational graphs** – using TensorBoard or other tools.

**Reflection:**
Understanding computational graphs is essential for working with deep learning frameworks. The ability to trace forward and backward passes is what enables gradient‑based optimisation. The shift from static to dynamic graphs has made deep learning more accessible and iterative, accelerating research and development.

# Building Computational Graphs from Scratch

This part demonstrates how to build a **computational graph** from scratch using Python classes. We define custom nodes that represent variables and operations (cosine, power, sum) and chain them together to form a graph. The graph is executed forward to compute the output, and we can visualise the graph structure.

**Key Concepts Covered:**
- Computational graph structure (nodes, edges, parent relationships).
- Forward propagation: evaluating the graph.
- Custom node classes for variables, unary operations, and binary operations.
- Chaining operations and building sequential graphs.
- Visualising the graph recursively.

**Key Techniques & Libraries:**
- `numpy` – numerical operations.
- Custom Python classes – representing nodes and operations.

**Objective:**
To understand the underlying mechanics of computational graphs, which are the foundation of automatic differentiation and deep learning frameworks.

In [1]:
import numpy as np

print("All imports successful.")

All imports successful.


## 1. Base Node Class

We define a base `Node` class that all other nodes inherit from. It stores:
- `require_grad` – whether gradients are needed (default False).
- `grad` – the gradient value (if required).
- `value` – the numerical value of the node (set during forward pass).

It also provides a `forward` method (to be overridden) and a string representation.

In [2]:
class Node:
    """Base class for all nodes in a computational graph."""
    def __init__(self, require_grad=False, grad=None):
        self.require_grad = require_grad
        self.grad = grad
        self.value = None

    def forward(self, *args):
        """To be overridden by subclasses."""
        raise NotImplementedError

    def __call__(self, *args):
        return self.forward(*args)

    def __str__(self):
        return str(self.value)

## 2. Input Node (Variable)

The `Variable` node holds input data. It can store a scalar or a NumPy array. It is marked as requiring a gradient (`require_grad=True`).

In [3]:
class Variable(Node):
    """Input node that holds a value (scalar or array)."""
    def __init__(self, value, array=False):
        super().__init__(require_grad=True)
        if array:
            self.value = np.array(value)
        else:
            self.value = value

    def forward(self):
        # No operation needed; value is already set
        return self

## 3. Operation Nodes

We define operation nodes that take one or more parent nodes and compute a result.

### 3.1 Cosine Function
The `Cos` node computes `cos(n * x)`, where `n` is a multiplier.

In [4]:
class Cos(Node):
    """Cosine function: cos(n * x)."""
    def __init__(self, n=2):
        super().__init__(require_grad=True)
        self.n = n

    def forward(self, x):
        self.value = np.cos(self.n * x.value)
        self.parent = x
        return self

### 3.2 Power Function
The `Pow` node raises its input to the power `p`.

In [5]:
class Pow(Node):
    """Power function: x**p."""
    def __init__(self, p=2):
        super().__init__(require_grad=True)
        self.pow = p

    def forward(self, x):
        self.value = x.value ** self.pow
        self.parent = x
        return self

### 3.3 Sum Function
The `Sum` node adds two input nodes.

In [6]:
class Sum(Node):
    """Addition of two nodes."""
    def __init__(self):
        super().__init__(require_grad=True)

    def forward(self, a, b):
        self.value = a.value + b.value
        self.parentA = a
        self.parentB = b
        return self

## 4. Using the Nodes to Build a Graph

We now create a computational graph that computes:

$f(x) = cos^4(2 cos^2(x)) + cos^2(x)$


We instantiate operation nodes and connect them by applying them to previous nodes.

In [7]:
# Create input variable
x = Variable(3)

# Create operation nodes
cos_op = Cos(n=1)
pow_op = Pow(p=2)
cos2_op = Cos(n=2)
pow4_op = Pow(p=4)
sum_op = Sum()

# Build the graph by chaining operations
n1 = cos_op(x)          # cos(x)
n2 = pow_op(n1)         # cos²(x)
n3 = cos2_op(n2)        # cos(2 * cos²(x))
n4 = pow4_op(n3)        # cos⁴(2 * cos²(x))
n5 = sum_op(n4, n2)     # cos⁴(...) + cos²(x)

print(f"Result of f(3): {n5.value:.6f}")

Result of f(3): 1.000851


## 5. Visualising the Graph

We define a recursive function to print the graph structure, showing the parent-child relationships.

In [8]:
def print_graph(node, indent='>'):
    """
    Recursively print the computational graph starting from the given node.
    """
    if isinstance(node, Variable):
        print(indent, f"Variable: {node.value}")
    elif isinstance(node, Sum):
        print(indent, f"Sum: {node.value}")
        print_graph(node.parentA, indent='\t' + indent)
        print_graph(node.parentB, indent='\t' + indent)
    elif isinstance(node, (Cos, Pow)):
        print(indent, f"{node.__class__.__name__}: {node.value}")
        print_graph(node.parent, indent='\t' + indent)
    else:
        print(indent, f"Unknown: {node.value}")

print_graph(n5)

> Sum: 1.0008508838138923
	> Pow: 0.020765740488709425
		> Cos: -0.37960931045656354
			> Pow: 0.9800851433251829
				> Cos: -0.9899924966004454
					> Variable: 3
	> Pow: 0.9800851433251829
		> Cos: -0.9899924966004454
			> Variable: 3


## 6. Sequential Graph Construction

For sequential computations (where one operation follows another), we can define a `Sequential` class that chains nodes automatically.

In [9]:
class Sequential(Node):
    """Chain multiple operations sequentially."""
    def __init__(self, *ops):
        super().__init__(require_grad=True)
        self.ops = ops

    def forward(self, x):
        self.value = x
        for op in self.ops:
            self.value = op(self.value)   # apply operation
        return self

    def print_graph(self):
        print_graph(self.ops[-1])  # print from the last node

### Example: f(x) = cos⁵(5 · cos²(2x))

We build a sequential graph for:

$f(x) = cos(5 (cos^2(2x)))^5$


The sequence of operations is:
1. Cos(n=2)
2. Pow(p=2)
3. Cos(n=5)
4. Pow(p=5)

In [11]:
# Define the sequential chain
y = Variable([2, 4], array=True)

seq = Sequential(
    Cos(n=2),   # cos(2x)
    Pow(p=2),   # cos^2(2x)
    Cos(n=5),   # cos(5 cos^2(2x))
    Pow(p=5)    # [cos(...)]^5
)

result = seq.forward(y)
print(f"Result of sequential graph: {result.value}")

print("\nGraph structure:")
seq.print_graph()

Result of sequential graph: [-0.04415795  0.97232642]

Graph structure:
> Pow: [-0.04415795  0.97232642]
	> Cos: [-0.53579886  0.99440298]
		> Pow: [0.42724998 0.02117026]
			> Cos: [-0.65364362 -0.14550003]
				> Variable: [2 4]


## Summary and Next Steps

**Accomplished:**
- Built a computational graph from scratch using custom Python classes.
- Created nodes for variables, cosine, power, and sum operations.
- Chained operations to form a graph and executed the forward pass.
- Implemented a recursive visualisation of the graph structure.
- Defined a `Sequential` class for simpler sequential graphs.

**Key Insights:**
- Computational graphs are directed acyclic graphs (DAGs) where nodes represent operations and edges represent data flow.
- Forward propagation evaluates the graph from input nodes to the final output.
- The graph structure is defined by parent-child relationships created during forward calls.
- This manual implementation illustrates the core concepts behind automatic differentiation frameworks (e.g., PyTorch, TensorFlow).

**Suggested Next Steps:**
1. **Add backward propagation** – implement the `backward` method for each node to compute gradients.
2. **Support more operations** – addition, multiplication, exponential, etc.
3. **Use this graph to train a simple model** – e.g., linear regression.

**Reflection:**
This notebook provides a bottom‑up understanding of computational graphs, which is essential for appreciating how deep learning frameworks perform automatic differentiation and optimisation. By building the graph ourselves, we demystify the internal workings of libraries like PyTorch.